# Karis's original OSRM walking-matrix builder (superseded)

This is the first part (cells 0 to 8) of Karis's `walking_network_analysis/osrm_walking_query.ipynb`,
kept as it was last run, with her saved outputs. It documents how the first block × destination
walking-time and distance matrices were built: OSRM graph preparation in Docker (walking on port
5001, driving on 5002), a `get_osrm_matrix` helper around the OSRM `table` service, and a chunked
loop over all 10,769 residential HDB centroids against 1,651 childcare centres (the deprecated
`data/Centres_2026.geojson`) and 613 MRT exits, about 11 minutes and 4,968 requests, saved as
`large_data/walk_matrix_time.json` / `walk_matrix_dist.json`.

**Superseded by** `1_Data_Preparation/1_06_build_walk_matrix.ipynb`, which runs the same OSRM table
queries against the canonical 1,796-centre layer (`output/childcare_centres_2026.gpkg`), plus the
planned centres and MRT exits as separate destination sets, and stores the results as parquet keyed
by `OBJECTID` and centre id (`large_data/walk_centres_time_min.parquet`, `walk_centres_dist_m.parquet`).

**Where the rest of her notebook went:** cells 9 to 21 (route sample on folium, matrix cleaning,
nearest-centre maps, 5/10/15-minute catchments) became `2_ESDA/2_04_walking_access.ipynb`, re-pointed
to the canonical matrices; cells 22 to 23 (Ripley's K for HDB blocks and centres) became
`2_ESDA/2_03_ripleys_k.ipynb`. Her geoplot KDE (cells 17 to 18) was replaced by the child-weighted
KDE in `2_ESDA/2_02_child_density_kde.ipynb`.

**Not executed here:** it needs her matrix files, the deprecated centre layer and a running OSRM
server; the outputs shown are from her last run. Do not run it to regenerate anything.


## OSRM Network Query

Key files needed: 

| File | Used for |
|---|---|
| output/hdb_blocks.gpkg | Residential points - my origins | 
| data/BusStops_2026.geojson | Bus stop points - additional destinations |
| data/MRTStationExits_2026.geojson | MRT exit points - additional destinations |
| data/Centres_2026.geojson | Childcare centres - destinations | 
| data/MasterPlanSubZones_2019.geojson | Study area |

Key Dependencies:

| Library | Purpose |
|---|---|
| `xlogit` | Model estimation |
| `pandas` | Tabular data manipulation |
| `matplotlib` | QA scatter plots |
| `pickle` | Save and export models |
| `numpy` | Data processing |
| `statsmodels` | |
| `seaborn` | Graphs for checking correlation between model variables|
| `folium` | Visualisation of routes|


The extracted OSRM Singapore files can also be directly downloaded here. 
Data for running OSRM via Docker:
https://www.dropbox.com/scl/fo/de4tvoq346ylpexiz2k2s/AAKRUqoSCLjPv95NMsHSMMM?rlkey=sddbtns26x7skqf0d8nh98yak&st=3ytafdb2&dl=0

But, if you prefer to extract the data yourself, please follow the manual extraction instructions:


--- 

### Manual Extraction

1. First, please obtain the OSRM files through running osrm-extract in Docker on the singapore_only osm.pbf file in your Shell/Terminal:

OSRM uses Lua profiles (such as car.lua, bicycle.lua, or foot.lua) to determine which roads are passable for a specific transport mode. Use the osrm-extract tool to parse the PBF data.

```bash
docker run -t -v "${PWD}:/data" osrm/osrm-backend osrm-extract -p /opt/car.lua /data/singapore.osm.pbf
```

(Replace with *walk* for the walking profiles)

2. Next, prepare the data for the Multi-Level Dijkstra (MLD) routing algorithm. This requires partitioning the map into cells and calculating routing weights.

#### Partition the graph into cells
```bash
docker run -t -v "${PWD}:/data" osrm/osrm-backend osrm-partition /data/singapore-latest.osrm`
```

#### Customize the cells with routing weights
```bash
docker run -t -v "${PWD}:/data" osrm/osrm-backend osrm-customize /data/singapore-latest.osrm`
```

### Local OSRM routing servers

Accessibility costs (\(C_{i,j}^{m}\)) are computed via local OSRM instances, one per mode. Each assumes the corresponding `.osrm` graph has already been built (`osrm-extract` → `osrm-partition` → `osrm-customize`) into the mounted data folder.

| Mode    | Port | Data folder       |
|---------|------|--------------------|
| Walking | 5001 | `OSRMData_Walk`    |
| Driving | 5002 | `OSRMData_Drive`   |



```bash
# Walking — port 5001
docker run --volume="<your_path>/OSRMData_Walk:/data" -p 5001:5000 -d osrm/osrm-backend
```

```bash
# Driving — port 5002
docker run --volume="<your_path>/OSRMData_Drive:/data" -p 5002:5000 -d osrm/osrm-backend
\`\`\`

# ─────────────────────────────────────────────

### Notes for BT4015 Analysis
How to calculate timings between each origin + destination?
###
Neighbourhood - its closest childcare centre 

In [4]:
#origin - destination matrix

def get_osrm_matrix(origins, destinations, mode="foot", metric="duration"):
    """Return an OSRM matrix: duration in minutes or distance in metres."""
    if metric not in ("duration", "distance"):
        raise ValueError("metric must be 'duration' or 'distance'")

    coords = origins + destinations
    coords_str = ";".join(f"{lon},{lat}" for lon, lat in coords)

    url = f"http://localhost:5002/table/v1/{mode}/{coords_str}"
    params = {
        "sources": ";".join(map(str, range(len(origins)))),
        "destinations": ";".join(map(str, range(len(origins), len(coords)))),
        "annotations": metric,
    }

    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    data = response.json()

    if data.get("code") != "Ok":
        raise RuntimeError(f"OSRM error: {data.get('message', data)}")

    key = "durations" if metric == "duration" else "distances"
    matrix = np.array(
        [
            [value if value is not None else np.inf for value in row]
            for row in data[key]
        ],
        dtype=float,
    )

    return matrix / 60 if metric == "duration" else matrix


def chunk_list(coords, chunk_size):
    """Yield coordinate batches of at most chunk_size."""
    for start in range(0, len(coords), chunk_size):
        yield coords[start:start + chunk_size]

### Calculating timings for walk matrix

Origins are HDBs; destinations combine childcare centres, bus stops and MRT exits. Childcare column labels are unchanged; transit column labels use `bus_stop:<source index>` and `mrt_exit:<source index>`. The loading cell prepares the points without querying OSRM. Run the matrix cell only when ready.

Matrix JSON files are saved in the gitignored `large_data` folder. Adding transit destinations increases the matrix size and number of requests. The childcare analyses below select only childcare columns.


In [5]:
#files
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
gpkg_path = "../output/hdb_blocks.gpkg"

hdbs = gpd.read_file(gpkg_path, layer="hdb_points")
ccs = gpd.read_file("../data/Centres_2026.geojson")

#bus_stops = gpd.read_file("../data/BusStops_2026.geojson")
mrt_exits = gpd.read_file("../data/MRTStationExits_2026.geojson")

# Keep childcare column labels for the childcare-only analyses below.
# Prefix transit labels to prevent collisions between source datasets.
destination_frames = []
for destination_type, points in [
    ("childcare", ccs), ("mrt_exit", mrt_exits), #"bus_stop", bus_stops),
]:
    points = points.to_crs(epsg=4326).copy()
    if (points.geometry.isna().any() or points.geometry.is_empty.any()
            or not points.geom_type.eq("Point").all()):
        raise ValueError(f"{destination_type} destinations must contain non-empty Point geometries.")
    points["destination_type"] = destination_type
    points["source_index"] = points.index
    if destination_type != "childcare":
        points.index = pd.Index([f"{destination_type}:{idx}" for idx in points.index])
    destination_frames.append(points)

gdf_origins = hdbs.to_crs(epsg=4326).copy()
gdf_destinations = gpd.GeoDataFrame(
    pd.concat(destination_frames), geometry="geometry", crs="EPSG:4326"
)
if not gdf_origins.index.is_unique or not gdf_destinations.index.is_unique:
    raise ValueError("Origin and destination labels must be unique.")

print(f"Loaded {len(hdbs):,} HDB origins and {len(gdf_destinations):,} destinations: "
      f"{len(ccs):,} childcare centres, {len(bus_stops):,} bus stops, "
      f"and {len(mrt_exits):,} MRT exits.")


Loaded 10,769 HDB origins and 2,264 destinations: 1,651 childcare centres, 5,205 bus stops, and 613 MRT exits.


In [6]:
from tqdm.auto import tqdm

# Origins (HDBs) and destinations (childcare, bus stops, MRT exits)
# are prepared in the loading cell.

gdf_origins = gdf_origins.to_crs(epsg=4326) #wgs84, osrm only works in global coordinates
gdf_destinations = gdf_destinations.to_crs(epsg=4326)

# OSRM coordinate order: (longitude, latitude)
coords_origins = list(zip(
    gdf_origins.geometry.x,
    gdf_origins.geometry.y
))
coords_destinations = list(zip(
    gdf_destinations.geometry.x,
    gdf_destinations.geometry.y
))

print(coords_origins[:3])
#should be around (103, 1.3)


#maximum chunk size for OSRM requests
max_size = 100

# crear una matriz completa llena de infinitos, de tamaño (num_origenes x num_destinos)
matrix_time = np.full((len(coords_origins), len(coords_destinations)), np.inf)
matrix_dist = np.full((len(coords_origins), len(coords_destinations)), np.inf)

# dividir orígenes y destinos en sub-listas
origen_chunks = list(chunk_list(coords_origins, max_size))
destino_chunks = list(chunk_list(coords_destinations, max_size))

num_origins = len(coords_origins)
num_destinations = len(coords_destinations)
total_queries = 2 * len(origen_chunks) * len(destino_chunks)
print(f"Origins: {num_origins:,} | Destinations: {num_destinations:,}")
print(f"Origin-destination pairs: {num_origins * num_destinations:,} | OSRM queries: {total_queries:,}")

# Each batch makes two requests: duration and distance.
with tqdm(total=total_queries, desc="OSRM queries", unit="query") as progress:
    for i, sub_origins in enumerate(origen_chunks):
        for j, sub_destinations in enumerate(destino_chunks):
            ini_i = i * max_size
            fin_i = ini_i + len(sub_origins)
            ini_j = j * max_size
            fin_j = ini_j + len(sub_destinations)
            progress.set_postfix_str(
                f"origins {ini_i + 1:,}-{fin_i:,}/{num_origins:,}, "
                f"destinations {ini_j + 1:,}-{fin_j:,}/{num_destinations:,}"
            )

            sub_matriz_t = get_osrm_matrix(sub_origins, sub_destinations, mode='foot', metric='duration')
            matrix_time[ini_i:fin_i, ini_j:fin_j] = sub_matriz_t
            progress.update(1)

            sub_matriz_d = get_osrm_matrix(sub_origins, sub_destinations, mode='foot', metric='distance')
            matrix_dist[ini_i:fin_i, ini_j:fin_j] = sub_matriz_d
            progress.update(1)

# convertir la matriz (numpy array) en un DataFrame de pandas, con index de orígenes y columns de destinos
walk_matrix_time = pd.DataFrame(matrix_time, index=gdf_origins.index, columns=gdf_destinations.index)
walk_matrix_dist = pd.DataFrame(matrix_dist, index=gdf_origins.index, columns=gdf_destinations.index)

# export matrix to json for use 
walk_matrix_time.to_json("../large_data/walk_matrix_time.json", orient="split")
walk_matrix_dist.to_json("../large_data/walk_matrix_dist.json", orient="split")

[(103.74684200757073, 1.3413230558229585), (103.76036105079291, 1.3856340613434799), (103.79346353096793, 1.307620048585763)]
Origins: 10,769 | Destinations: 2,264
Origin-destination pairs: 24,381,016 | OSRM queries: 4,968


OSRM queries: 100%|██████████| 4968/4968 [10:43<00:00,  7.72query/s, origins 10,701-10,769/10,769, destinations 2,201-2,264/2,264] 


# ─────────────────────────────────────────────